# Advanced Pandas Interview Prep

This notebook covers deep Pandas concepts frequently tested in data science interviews at top companies. Each section includes working code, interview questions with answers, and common gotchas.

**Topics covered:**
1. loc vs iloc vs at vs iat vs [] operator
2. SettingWithCopyWarning deep dive
3. apply vs applymap vs map vs vectorized ops
4. groupby internals
5. MultiIndex and stack/unstack
6. merge types deep dive
7. Window functions (rolling, expanding, ewm)
8. Memory optimization
9. pd.cut vs pd.qcut
10. pipe() method and method chaining
11. eval() and query()
12. Handling datetime
13. explode(), melt(), wide_to_long()
14. Duplicates deep dive
15. Common interview traps

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')  # We'll enable selectively below

print(f'pandas version: {pd.__version__}')
print(f'numpy version: {np.__version__}')

pandas version: 3.0.5
numpy version: 2.4.6


---
## 1. loc vs iloc vs at vs iat vs [] operator

| Accessor | Index Type | Returns | Speed |
|----------|-----------|---------|-------|
| `[]` | label (columns) or label/bool (rows) | Series or DataFrame | fast |
| `.loc[]` | label-based | scalar, Series, DataFrame | medium |
| `.iloc[]` | integer position | scalar, Series, DataFrame | medium |
| `.at[]` | label-based | scalar only | fastest |
| `.iat[]` | integer position | scalar only | fastest |

In [2]:
# Setup: realistic employee DataFrame
np.random.seed(42)
df = pd.DataFrame({
    'name':   ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'dept':   ['Eng', 'Eng', 'HR', 'HR', 'Eng'],
    'salary': [95000, 85000, 72000, 68000, 110000],
    'years':  [5, 3, 8, 2, 10]
}, index=['e1', 'e2', 'e3', 'e4', 'e5'])

print(df)

     name dept  salary  years
e1  Alice  Eng   95000      5
e2    Bob  Eng   85000      3
e3  Carol   HR   72000      8
e4   Dave   HR   68000      2
e5    Eve  Eng  110000     10


In [3]:
# --- [] operator ---
# Single string: selects COLUMN by label
print('df["salary"]:')
print(df['salary'])
print()

# Slice on [] operator selects ROWS by label (if string index)
# But with integer index it selects by position — this is inconsistent!
print('df["e1":"e3"] (row slice by label):')
print(df['e1':'e3'])  # label slice, inclusive on both ends

df["salary"]:
e1     95000
e2     85000
e3     72000
e4     68000
e5    110000
Name: salary, dtype: int64

df["e1":"e3"] (row slice by label):
     name dept  salary  years
e1  Alice  Eng   95000      5
e2    Bob  Eng   85000      3
e3  Carol   HR   72000      8


In [4]:
# --- .loc[] — label based, INCLUSIVE on both ends ---
print('df.loc["e2", "salary"]:', df.loc['e2', 'salary'])
print()
print('df.loc["e1":"e3", "name":"salary"]:')
print(df.loc['e1':'e3', 'name':'salary'])
print()
# Boolean indexing with loc
print('df.loc[df["salary"] > 80000]:')
print(df.loc[df['salary'] > 80000])

df.loc["e2", "salary"]: 85000

df.loc["e1":"e3", "name":"salary"]:
     name dept  salary
e1  Alice  Eng   95000
e2    Bob  Eng   85000
e3  Carol   HR   72000

df.loc[df["salary"] > 80000]:
     name dept  salary  years
e1  Alice  Eng   95000      5
e2    Bob  Eng   85000      3
e5    Eve  Eng  110000     10


In [5]:
# --- .iloc[] — integer position, EXCLUSIVE on right end (like Python slices) ---
print('df.iloc[1, 2]:', df.iloc[1, 2])       # row 1, col 2
print()
print('df.iloc[0:3, 0:2]:')
print(df.iloc[0:3, 0:2])                      # rows 0,1,2 — cols 0,1 (NOT col 2)
print()
print('df.iloc[-1]:')
print(df.iloc[-1])                            # last row

df.iloc[1, 2]: 85000

df.iloc[0:3, 0:2]:
     name dept
e1  Alice  Eng
e2    Bob  Eng
e3  Carol   HR

df.iloc[-1]:
name         Eve
dept         Eng
salary    110000
years         10
Name: e5, dtype: object


In [6]:
# --- .at[] and .iat[] — scalar access, fastest for single values ---
import timeit

# .at uses labels
print('df.at["e3", "salary"]:', df.at['e3', 'salary'])

# .iat uses integer positions
print('df.iat[2, 2]:', df.iat[2, 2])

# Speed comparison
t_loc  = timeit.timeit(lambda: df.loc['e3', 'salary'],  number=10000)
t_at   = timeit.timeit(lambda: df.at['e3', 'salary'],   number=10000)
t_iat  = timeit.timeit(lambda: df.iat[2, 2],            number=10000)
t_iloc = timeit.timeit(lambda: df.iloc[2, 2],           number=10000)

print(f'\nSpeed (10k calls): loc={t_loc:.3f}s  at={t_at:.3f}s  iloc={t_iloc:.3f}s  iat={t_iat:.3f}s')
print('at/iat are fastest for scalar access')

df.at["e3", "salary"]: 72000
df.iat[2, 2]: 72000



Speed (10k calls): loc=0.225s  at=0.167s  iloc=0.186s  iat=0.140s
at/iat are fastest for scalar access


In [7]:
# CRITICAL: loc vs iloc difference when index is integers
df_int = pd.DataFrame({'val': [10, 20, 30]}, index=[5, 10, 15])

print('Index:', df_int.index.tolist())
print()
print('df_int.loc[5]:  ', df_int.loc[5, 'val'])    # label 5 → row with index=5 → val=10
print('df_int.iloc[0]: ', df_int.iloc[0, 0])        # position 0 → first row → val=10
print()
print('df_int.loc[10]: ', df_int.loc[10, 'val'])   # label 10 → val=20
print('df_int.iloc[1]: ', df_int.iloc[1, 0])        # position 1 → val=20
print()
print('df_int.loc[15]: ', df_int.loc[15, 'val'])   # label 15 → val=30
print('df_int.iloc[2]: ', df_int.iloc[2, 0])        # position 2 → val=30

Index: [5, 10, 15]

df_int.loc[5]:   10
df_int.iloc[0]:  10

df_int.loc[10]:  20
df_int.iloc[1]:  20

df_int.loc[15]:  30
df_int.iloc[2]:  30


> **INTERVIEW QUESTION:** What is the difference between `df.loc[5]` and `df.iloc[5]` when the DataFrame has an integer index that does not start at 0?
>
> **Answer:** `.loc[5]` selects the row whose **index label** equals 5. `.iloc[5]` selects the row at **integer position** 5 (the 6th row regardless of label). If the index is `[10, 20, 30, ...]`, `df.loc[5]` raises `KeyError`, while `df.iloc[5]` returns the 6th row.

> **PANDAS GOTCHA:** The `[]` operator is **inconsistent** — `df['col']` selects a column, but `df[0:3]` selects rows. This asymmetry trips up beginners. Always prefer `.loc`/`.iloc` for explicit row/column selection.

---
## 2. SettingWithCopyWarning Deep Dive

This is one of the most common Pandas pitfalls. It happens because **chained indexing** creates ambiguity about whether you're modifying the original DataFrame or a copy.

In [8]:
import warnings
warnings.filterwarnings('error', category=pd.errors.SettingWithCopyWarning if hasattr(pd.errors, "SettingWithCopyWarning") else Warning)

df = pd.DataFrame({
    'dept':   ['Eng', 'Eng', 'HR', 'HR', 'Eng'],
    'salary': [95000, 85000, 72000, 68000, 110000],
    'bonus':  [0, 0, 0, 0, 0]
})

# --- THE BUG: chained indexing ---
# Step 1: df[df['dept']=='Eng']  may return a VIEW or a COPY (Pandas decides)
# Step 2: ['bonus'] = 5000  modifies... what? Possibly a temporary copy!
try:
    df[df['dept'] == 'Eng']['bonus'] = 5000   # SettingWithCopyWarning!
except Exception as e:
    print(f'Warning raised: {type(e).__name__}')

print('\nbonus column after chained assignment:')
print(df['bonus'].tolist())  # Still all zeros — the assignment was silently lost!

Warning raised: ChainedAssignmentError

bonus column after chained assignment:
[0, 0, 0, 0, 0]


In [9]:
warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning if hasattr(pd.errors, "SettingWithCopyWarning") else Warning)

# --- THE FIX 1: use .loc with boolean mask in a single step ---
df.loc[df['dept'] == 'Eng', 'bonus'] = 5000
print('After .loc fix:')
print(df[['dept', 'bonus']])

After .loc fix:
  dept  bonus
0  Eng   5000
1  Eng   5000
2   HR      0
3   HR      0
4  Eng   5000


In [10]:
# --- THE FIX 2: explicit .copy() when you WANT a separate object ---
df2 = df.copy()
eng_df = df2[df2['dept'] == 'Eng'].copy()  # explicit copy
eng_df['bonus'] = 10000  # Now this is fine — eng_df is definitively a separate object

print('eng_df (copy):')
print(eng_df[['dept', 'bonus']])
print()
print('df2 unchanged:')
print(df2[['dept', 'bonus']])

eng_df (copy):
  dept  bonus
0  Eng  10000
1  Eng  10000
4  Eng  10000

df2 unchanged:
  dept  bonus
0  Eng   5000
1  Eng   5000
2   HR      0
3   HR      0
4  Eng   5000


In [11]:
# --- UNDERSTANDING VIEWS vs COPIES ---
# Slicing by rows usually creates a VIEW (shares memory)
# Fancy indexing (boolean mask, list of labels) usually creates a COPY

df3 = pd.DataFrame({'a': [1, 2, 3, 4, 5], 'b': [10, 20, 30, 40, 50]})

# Row slice → likely a view
slice_view = df3.iloc[1:4]
print('shares memory (slice)?', np.shares_memory(df3['a'], slice_view['a']))  # often True

# Boolean mask → always a copy
bool_copy = df3[df3['a'] > 2]
print('shares memory (bool)?', np.shares_memory(df3['a'], bool_copy['a']))    # always False

# List indexing → always a copy
list_copy = df3.loc[[1, 2, 3]]
print('shares memory (list)?', np.shares_memory(df3['a'], list_copy['a']))    # always False

shares memory (slice)? True
shares memory (bool)? False
shares memory (list)? False


In [12]:
# --- FALSE ALARM vs REAL BUG ---
# False alarm: you DON'T intend to propagate changes back to original
df4 = pd.DataFrame({'x': range(5), 'y': range(5, 10)})
subset = df4[df4['x'] > 2].copy()  # .copy() silences the warning AND is correct
subset['z'] = 99  # You only want to modify subset — this is fine
print('subset with new col:')
print(subset)
print('df4 unchanged:')
print(df4)

# Real bug: chained assignment where you INTEND to modify original
# Use .loc to fix

subset with new col:
   x  y   z
3  3  8  99
4  4  9  99
df4 unchanged:
   x  y
0  0  5
1  1  6
2  2  7
3  3  8
4  4  9


> **INTERVIEW QUESTION:** What causes `SettingWithCopyWarning` and how do you fix it?
>
> **Answer:** It's caused by **chained indexing** — two consecutive `[]` operations like `df[condition]['col'] = val`. Pandas cannot guarantee whether the intermediate object is a view (shares memory with original) or a copy. The fix depends on intent:
> - **Modifying the original:** use `.loc` in a single step: `df.loc[condition, 'col'] = val`
> - **Working on a separate subset:** call `.copy()` explicitly: `subset = df[condition].copy()`
>
> **PANDAS GOTCHA:** The warning is non-deterministic — the same code may work sometimes (when Pandas returns a view) and silently fail other times (when it returns a copy). Never ignore it.

---
## 3. apply vs applymap vs map vs vectorized

Performance ranking (fastest to slowest):
1. **Vectorized numpy/pandas ops** — operates on entire arrays, C-speed
2. **`str`/`dt` accessor methods** — vectorized under the hood
3. **`Series.map()`** — element-wise on a Series, can use dict/Series lookup
4. **`DataFrame.apply()`** — applies function along axis, uses Python loop per row/col
5. **`DataFrame.applymap()` / `DataFrame.map()`** — element-wise on every cell, slowest

In [13]:
import time

np.random.seed(0)
N = 100_000
df_perf = pd.DataFrame({
    'a': np.random.randn(N),
    'b': np.random.randint(1, 100, N)
})

# Task: square column 'a'

def time_it(label, fn, reps=5):
    t = time.perf_counter()
    for _ in range(reps): fn()
    elapsed = (time.perf_counter() - t) / reps * 1000
    print(f'{label:40s}: {elapsed:.2f} ms')

time_it('vectorized (** 2)',        lambda: df_perf['a'] ** 2)
time_it('numpy np.square',          lambda: np.square(df_perf['a']))
time_it('Series.map(lambda)',        lambda: df_perf['a'].map(lambda x: x**2))
time_it('Series.apply(lambda)',      lambda: df_perf['a'].apply(lambda x: x**2))
time_it('DataFrame.apply(row-wise)', lambda: df_perf.apply(lambda row: row['a']**2, axis=1))

vectorized (** 2)                       : 0.32 ms
numpy np.square                         : 0.16 ms


Series.map(lambda)                      : 37.06 ms


Series.apply(lambda)                    : 34.72 ms


DataFrame.apply(row-wise)               : 564.96 ms


In [14]:
# --- Series.map() — element-wise, also great for dict/Series lookups ---
s = pd.Series(['cat', 'dog', 'bird', 'cat', 'dog'])

# map with dict (fast lookup — O(1) per element)
animal_map = {'cat': 'feline', 'dog': 'canine', 'bird': 'avian'}
print('map with dict:')
print(s.map(animal_map))
print()

# map with function
print('map with function:')
print(s.map(str.upper))

map with dict:
0    feline
1    canine
2     avian
3    feline
4    canine
dtype: str

map with function:
0     CAT
1     DOG
2    BIRD
3     CAT
4     DOG
dtype: str


In [15]:
# --- DataFrame.apply() — useful for multi-column logic ---
df_emp = pd.DataFrame({
    'salary': [95000, 85000, 72000, 68000, 110000],
    'years':  [5, 3, 8, 2, 10]
})

# apply along axis=1 (per row) — computes bonus based on multiple cols
df_emp['bonus'] = df_emp.apply(
    lambda row: row['salary'] * 0.1 if row['years'] >= 5 else row['salary'] * 0.05,
    axis=1
)
print('With computed bonus:')
print(df_emp)

# Better: replace with vectorized np.where
df_emp['bonus_v'] = np.where(df_emp['years'] >= 5,
                              df_emp['salary'] * 0.1,
                              df_emp['salary'] * 0.05)
print('\nResults identical:', (df_emp['bonus'] == df_emp['bonus_v']).all())

With computed bonus:
   salary  years    bonus
0   95000      5   9500.0
1   85000      3   4250.0
2   72000      8   7200.0
3   68000      2   3400.0
4  110000     10  11000.0

Results identical: True


In [16]:
# --- applymap / DataFrame.map (pandas >= 2.1) --- element-wise on every cell
df_small = pd.DataFrame({'x': [1.1, 2.2, 3.3], 'y': [4.4, 5.5, 6.6]})

# Old API: applymap (deprecated in pandas 2.1+)
# New API: DataFrame.map()
try:
    result = df_small.map(lambda v: round(v, 1))  # pandas >= 2.1
except AttributeError:
    result = df_small.applymap(lambda v: round(v, 1))  # pandas < 2.1

print('Element-wise round:')
print(result)

# BETTER: vectorized
print('\nVectorized round:')
print(df_small.round(1))

Element-wise round:
     x    y
0  1.1  4.4
1  2.2  5.5
2  3.3  6.6

Vectorized round:
     x    y
0  1.1  4.4
1  2.2  5.5
2  3.3  6.6


In [17]:
# --- When apply() is legitimately the right tool ---
# 1. When you need a complex multi-output per-row transformation
df2 = pd.DataFrame({'text': ['hello world', 'foo bar baz', 'one']})
df2[['word_count', 'char_count']] = df2['text'].apply(
    lambda t: pd.Series([len(t.split()), len(t)])
)
print(df2)

# 2. When applying a grouped reduction that isn't built-in
# (better use groupby + agg — see section 4)

          text  word_count  char_count
0  hello world           2          11
1  foo bar baz           3          11
2          one           1           3


> **INTERVIEW QUESTION:** When would you choose `apply()` over a vectorized operation, and why is vectorized always preferred if possible?
>
> **Answer:** Vectorized operations (NumPy ufuncs, pandas built-ins) operate on entire arrays in C, avoiding Python interpreter overhead. `apply()` calls a Python function once per row/element, which can be 10-100x slower on large datasets. Use `apply()` only when the logic is genuinely complex, multi-column, or returns a non-scalar (like a `pd.Series`). Replace single-condition logic with `np.where`; replace string ops with the `.str` accessor; replace math ops with direct arithmetic.
>
> **PANDAS GOTCHA:** `Series.map()` and `Series.apply()` have nearly identical performance (both call Python per element), but `map()` is faster with dict-lookup and skips `NaN` by default. `DataFrame.apply(axis=1)` is the slow row-wise loop — profile before using on large data.

---
## 4. groupby Internals — split-apply-combine

The groupby mechanism:
1. **Split**: partition the DataFrame into groups by key(s)
2. **Apply**: run a function on each group
3. **Combine**: concatenate results back into a single object

In [18]:
np.random.seed(1)
df_sales = pd.DataFrame({
    'region':  np.random.choice(['North', 'South', 'East', 'West'], 20),
    'product': np.random.choice(['A', 'B', 'C'], 20),
    'revenue': np.random.randint(1000, 10000, 20),
    'units':   np.random.randint(10, 200, 20),
    'quarter': np.random.choice(['Q1', 'Q2', 'Q3', 'Q4'], 20)
})
print(df_sales.head(8))

  region product  revenue  units quarter
0  South       C     4606     19      Q3
1   West       B     3561    136      Q3
2  North       C     9920     33      Q1
3  North       A     7771    135      Q1
4   West       A     1431    110      Q4
5  South       C     5074    165      Q4
6   West       A     1542    175      Q2
7  South       B     2478     67      Q2


In [19]:
# --- agg: reduce each group to scalar(s) ---
# Simple
print('Mean revenue by region:')
print(df_sales.groupby('region')['revenue'].mean())
print()

# Multiple functions
print('Multiple aggregations:')
print(df_sales.groupby('region')['revenue'].agg(['mean', 'sum', 'std', 'count']))

Mean revenue by region:
region
East     2775.000000
North    6092.428571
South    4557.500000
West     2343.000000
Name: revenue, dtype: float64

Multiple aggregations:
               mean    sum          std  count
region                                        
East    2775.000000   5550   663.266161      2
North   6092.428571  42647  3665.091398      7
South   4557.500000  27345  2369.696162      6
West    2343.000000  11715   895.531406      5


In [20]:
# --- Named aggregations (pandas >= 0.25) --- clean, explicit, no MultiIndex columns ---
result = df_sales.groupby('region').agg(
    avg_revenue   = ('revenue', 'mean'),
    total_revenue = ('revenue', 'sum'),
    avg_units     = ('units',   'mean'),
    num_deals     = ('revenue', 'count')
)
print('Named aggregations:')
print(result.round(1))

Named aggregations:
        avg_revenue  total_revenue  avg_units  num_deals
region                                                  
East         2775.0           5550       99.5          2
North        6092.4          42647      128.4          7
South        4557.5          27345       60.2          6
West         2343.0          11715      106.8          5


In [21]:
# --- transform: returns same shape as input (for adding group stats back) ---
# Classic use case: subtract group mean (normalization)
df_sales['revenue_zscore'] = df_sales.groupby('region')['revenue'].transform(
    lambda x: (x - x.mean()) / x.std()
)
print('Revenue z-score by region:')
print(df_sales[['region', 'revenue', 'revenue_zscore']].head(8).round(3))

Revenue z-score by region:
  region  revenue  revenue_zscore
0  South     4606           0.020
1   West     3561           1.360
2  North     9920           1.044
3  North     7771           0.458
4   West     1431          -1.018
5  South     5074           0.218
6   West     1542          -0.894
7  South     2478          -0.878


In [22]:
# transform with built-in string name is much faster than lambda
df_sales['region_avg'] = df_sales.groupby('region')['revenue'].transform('mean')
df_sales['region_rank'] = df_sales.groupby('region')['revenue'].rank(ascending=False)
print(df_sales[['region', 'revenue', 'region_avg', 'region_rank']].head(8).round(1))

  region  revenue  region_avg  region_rank
0  South     4606      4557.5          3.0
1   West     3561      2343.0          1.0
2  North     9920      6092.4          1.0
3  North     7771      6092.4          4.0
4   West     1431      2343.0          5.0
5  South     5074      4557.5          2.0
6   West     1542      2343.0          4.0
7  South     2478      4557.5          6.0


In [23]:
# --- apply on groupby: most flexible, returns anything ---
# Use when agg/transform don't fit (e.g., top-N per group)
def top2(group):
    return group.nlargest(2, 'revenue')

top2_per_region = df_sales.groupby('region', group_keys=False).apply(top2)
print('Top 2 deals per region:')
# In pandas 2.2+, groupby key is not passed to apply - reset index to get region back
result = top2_per_region.reset_index()
if 'region' not in result.columns:
    result = top2_per_region.reset_index(level=0)
print(result[['region', 'product', 'revenue']].sort_values('region') if 'region' in result.columns else top2_per_region[['product', 'revenue']])


Top 2 deals per region:
   product  revenue
16       B     3244
18       B     2306
2        C     9920
10       A     9689
17       B     8906
5        C     5074
1        B     3561
13       C     2844


In [24]:
# --- filter: keep groups that satisfy a condition ---
# Keep only regions with average revenue > 5000
high_rev = df_sales.groupby('region').filter(lambda g: g['revenue'].mean() > 5000)
print('Regions with avg revenue > 5000:')
print(high_rev['region'].unique())
print(f'Rows kept: {len(high_rev)} / {len(df_sales)}')

Regions with avg revenue > 5000:
<StringArray>
['North']
Length: 1, dtype: str
Rows kept: 7 / 20


In [25]:
# --- agg vs transform vs apply summary ---
print('agg:       reduces group → scalar per group')
print('transform: same shape as input, group stats broadcast back')
print('apply:     most flexible, arbitrary return shape')
print('filter:    returns rows from groups that pass predicate')

# Gotcha: apply is slowest; prefer agg/transform with built-in functions
import time

g = df_sales.groupby('region')['revenue']

t1 = time.perf_counter()
for _ in range(1000): g.transform('mean')
t2 = time.perf_counter()
for _ in range(1000): g.transform(lambda x: x.mean())
t3 = time.perf_counter()

print(f'\ntransform("mean"): {(t2-t1)*1000:.1f}ms  vs  transform(lambda): {(t3-t2)*1000:.1f}ms')

agg:       reduces group → scalar per group
transform: same shape as input, group stats broadcast back
apply:     most flexible, arbitrary return shape
filter:    returns rows from groups that pass predicate



transform("mean"): 146.4ms  vs  transform(lambda): 836.3ms


> **INTERVIEW QUESTION:** What is the difference between `groupby().agg()`, `groupby().transform()`, and `groupby().apply()`?
>
> **Answer:**
> - `agg()`: **reduces** each group to one scalar. Output has one row per group. Use for summary statistics.
> - `transform()`: returns an object with the **same index as input** — group-level results are broadcast back to each row. Use to add group statistics as new columns (e.g., group mean for normalization).
> - `apply()`: **most flexible** — the function can return a scalar, Series, or DataFrame. Use for complex per-group logic like top-N rows.
>
> **PANDAS GOTCHA:** Using `apply` with a lambda that just calls a built-in (e.g., `lambda x: x.mean()`) is 5-10x slower than passing the string directly (`'mean'`). Always use string/named aggregations when possible.

---
## 5. MultiIndex and stack/unstack

In [26]:
# Creating MultiIndex DataFrames
arrays = [
    ['2023', '2023', '2023', '2024', '2024', '2024'],
    ['Q1',   'Q2',   'Q3',   'Q1',   'Q2',   'Q3']
]
mi = pd.MultiIndex.from_arrays(arrays, names=['year', 'quarter'])

df_mi = pd.DataFrame({
    'revenue': [100, 120, 130, 115, 135, 150],
    'cost':    [60,  70,  75,  65,  80,  90]
}, index=mi)

print(df_mi)

              revenue  cost
year quarter               
2023 Q1           100    60
     Q2           120    70
     Q3           130    75
2024 Q1           115    65
     Q2           135    80
     Q3           150    90


In [27]:
# --- Slicing MultiIndex with .loc ---
print('All of 2023:')
print(df_mi.loc['2023'])
print()
print('2023 Q2:')
print(df_mi.loc[('2023', 'Q2')])

All of 2023:
         revenue  cost
quarter               
Q1           100    60
Q2           120    70
Q3           130    75

2023 Q2:
revenue    120
cost        70
Name: (2023, Q2), dtype: int64


In [28]:
# --- xs() — cross-section — cleaner for inner level slicing ---
print('xs() — all Q1 rows across years:')
print(df_mi.xs('Q1', level='quarter'))
print()
print('xs() with drop_level=False:')
print(df_mi.xs('Q1', level='quarter', drop_level=False))

xs() — all Q1 rows across years:
      revenue  cost
year               
2023      100    60
2024      115    65

xs() with drop_level=False:
              revenue  cost
year quarter               
2023 Q1           100    60
2024 Q1           115    65


In [29]:
# --- stack / unstack ---
# unstack: move an index level INTO columns
df_unstacked = df_mi['revenue'].unstack(level='quarter')
print('unstacked (quarter → columns):')
print(df_unstacked)
print()

# stack: move a column level INTO rows (inverse of unstack)
df_stacked_back = df_unstacked.stack()
print('stacked back:')
print(df_stacked_back)

unstacked (quarter → columns):
quarter   Q1   Q2   Q3
year                  
2023     100  120  130
2024     115  135  150

stacked back:
year  quarter
2023  Q1         100
      Q2         120
      Q3         130
2024  Q1         115
      Q2         135
      Q3         150
dtype: int64


In [30]:
# --- Creating MultiIndex from groupby ---
df_grouped = df_sales.groupby(['region', 'product'])['revenue'].agg(['sum', 'mean'])
print('MultiIndex from groupby:')
print(df_grouped.head(8))
print()
print('Index type:', type(df_grouped.index))

MultiIndex from groupby:
                  sum         mean
region product                    
East   B         5550  2775.000000
North  A        17460  8730.000000
       B         6144  3072.000000
       C        19043  6347.666667
South  A         3669  3669.000000
       B        13996  4665.333333
       C         9680  4840.000000
West   A         2973  1486.500000

Index type: <class 'pandas.MultiIndex'>


In [31]:
# --- swaplevel + sort_index ---
df_swapped = df_grouped.swaplevel().sort_index()
print('After swaplevel (product first):')
print(df_swapped.head(8))

After swaplevel (product first):
                  sum         mean
product region                    
A       North   17460  8730.000000
        South    3669  3669.000000
        West     2973  1486.500000
B       East     5550  2775.000000
        North    6144  3072.000000
        South   13996  4665.333333
        West     3561  3561.000000
C       North   19043  6347.666667


In [32]:
# --- reset_index: flatten MultiIndex back to regular columns ---
df_flat = df_grouped.reset_index()
print('After reset_index:')
print(df_flat.head(6))

# Flatten MultiIndex columns (from pivot_table/unstack)
df_pivot = df_sales.pivot_table(values='revenue', index='region', columns='product', aggfunc='sum')
df_pivot.columns = ['_'.join(col).strip() for col in df_pivot.columns]
print('\nFlattened pivot columns:')
print(df_pivot.columns.tolist())

After reset_index:
  region product    sum         mean
0   East       B   5550  2775.000000
1  North       A  17460  8730.000000
2  North       B   6144  3072.000000
3  North       C  19043  6347.666667
4  South       A   3669  3669.000000
5  South       B  13996  4665.333333

Flattened pivot columns:
['A', 'B', 'C']


> **INTERVIEW QUESTION:** When would you use `unstack()` vs `pivot_table()`?
>
> **Answer:** Both reshape long-form data to wide-form, but:
> - `unstack()` works directly on a MultiIndex — it moves an existing index level into columns. Use after `groupby()` produces a MultiIndex.
> - `pivot_table()` works from scratch on regular columns — you specify index, columns, values, and aggregation. It's more readable for ad-hoc reshaping and handles duplicates by aggregating.
>
> **PANDAS GOTCHA:** After `groupby + unstack`, you often get `NaN` where combinations don't exist. Pass `fill_value=0` to `unstack()` to handle this.

---
## 6. merge Types Deep Dive

In [33]:
# Setup realistic tables
employees = pd.DataFrame({
    'emp_id':   [1, 2, 3, 4, 5],
    'name':     ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'dept_id':  [10, 10, 20, 20, 30]    # dept 30 has no name in dept table
})

departments = pd.DataFrame({
    'dept_id':  [10, 20, 40],           # dept 40 has no employees
    'dept_name': ['Engineering', 'HR', 'Marketing']
})

print('employees:')
print(employees)
print()
print('departments:')
print(departments)

employees:
   emp_id   name  dept_id
0       1  Alice       10
1       2    Bob       10
2       3  Carol       20
3       4   Dave       20
4       5    Eve       30

departments:
   dept_id    dept_name
0       10  Engineering
1       20           HR
2       40    Marketing


In [34]:
# INNER: only matching rows in both
inner = employees.merge(departments, on='dept_id', how='inner')
print('INNER join (5 employees, 2 dept matches → 4 rows):')
print(inner)
print()

INNER join (5 employees, 2 dept matches → 4 rows):
   emp_id   name  dept_id    dept_name
0       1  Alice       10  Engineering
1       2    Bob       10  Engineering
2       3  Carol       20           HR
3       4   Dave       20           HR



In [35]:
# LEFT: all left rows, NaN where no right match
left = employees.merge(departments, on='dept_id', how='left')
print('LEFT join (all 5 employees, dept 30 → NaN):')
print(left)
print()

LEFT join (all 5 employees, dept 30 → NaN):
   emp_id   name  dept_id    dept_name
0       1  Alice       10  Engineering
1       2    Bob       10  Engineering
2       3  Carol       20           HR
3       4   Dave       20           HR
4       5    Eve       30          NaN



In [36]:
# RIGHT: all right rows
right = employees.merge(departments, on='dept_id', how='right')
print('RIGHT join (all 3 depts, dept 40 → NaN):')
print(right)
print()

RIGHT join (all 3 depts, dept 40 → NaN):
   emp_id   name  dept_id    dept_name
0     1.0  Alice       10  Engineering
1     2.0    Bob       10  Engineering
2     3.0  Carol       20           HR
3     4.0   Dave       20           HR
4     NaN    NaN       40    Marketing



In [37]:
# OUTER: all rows from both
outer = employees.merge(departments, on='dept_id', how='outer')
print('OUTER join (all rows, NaN where unmatched):')
print(outer)
print()

OUTER join (all rows, NaN where unmatched):
   emp_id   name  dept_id    dept_name
0     1.0  Alice       10  Engineering
1     2.0    Bob       10  Engineering
2     3.0  Carol       20           HR
3     4.0   Dave       20           HR
4     5.0    Eve       30          NaN
5     NaN    NaN       40    Marketing



In [38]:
# CROSS: cartesian product (pandas >= 1.2)
cross = employees[['emp_id', 'name']].merge(departments[['dept_name']], how='cross')
print(f'CROSS join: {len(employees)} × {len(departments)} = {len(cross)} rows')
print(cross.head(6))

CROSS join: 5 × 3 = 15 rows
   emp_id   name    dept_name
0       1  Alice  Engineering
1       1  Alice           HR
2       1  Alice    Marketing
3       2    Bob  Engineering
4       2    Bob           HR
5       2    Bob    Marketing


In [39]:
# --- indicator=True — debug which rows matched ---
left_ind = employees.merge(departments, on='dept_id', how='left', indicator=True)
print('With indicator column:')
print(left_ind[['name', 'dept_id', '_merge']])
print()
print('Value counts:')
print(left_ind['_merge'].value_counts())

With indicator column:
    name  dept_id     _merge
0  Alice       10       both
1    Bob       10       both
2  Carol       20       both
3   Dave       20       both
4    Eve       30  left_only

Value counts:
_merge
both          4
left_only     1
right_only    0
Name: count, dtype: int64


In [40]:
# --- Handling duplicate column names with suffixes ---
df_a = pd.DataFrame({'id': [1,2,3], 'value': [10,20,30], 'ts': ['2023-01', '2023-02', '2023-03']})
df_b = pd.DataFrame({'id': [1,2,3], 'value': [100,200,300], 'ts': ['2023-01', '2023-02', '2023-03']})

merged = df_a.merge(df_b, on='id', suffixes=('_a', '_b'))
print('Custom suffixes:')
print(merged)

Custom suffixes:
   id  value_a     ts_a  value_b     ts_b
0   1       10  2023-01      100  2023-01
1   2       20  2023-02      200  2023-02
2   3       30  2023-03      300  2023-03


In [41]:
# --- left_on / right_on when key names differ ---
orders = pd.DataFrame({'order_id': [101,102,103], 'customer': [1,2,1]})
customers = pd.DataFrame({'cust_id': [1,2,3], 'cust_name': ['Alice','Bob','Carol']})

merged2 = orders.merge(customers, left_on='customer', right_on='cust_id')
print('Merge with different key names:')
print(merged2)

Merge with different key names:
   order_id  customer  cust_id cust_name
0       101         1        1     Alice
1       102         2        2       Bob
2       103         1        1     Alice


In [42]:
# --- validate parameter — catches unexpected duplicates ---
try:
    employees.merge(departments, on='dept_id', how='inner', validate='m:1')
    print('m:1 validation passed — each dept_id appears once in departments')
except pd.errors.MergeError as e:
    print(f'MergeError: {e}')

# Force a duplicate to show the error
dept_dup = pd.concat([departments, departments.iloc[[0]]])
try:
    employees.merge(dept_dup, on='dept_id', how='inner', validate='m:1')
except pd.errors.MergeError as e:
    print(f'\nWith duplicate in departments:')
    print(f'MergeError raised: {e}')

m:1 validation passed — each dept_id appears once in departments

With duplicate in departments:
MergeError raised: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
  dept_id
      10 ...


In [43]:
# --- merge vs join vs concat ---
# merge: SQL-style join on columns or index, most flexible
# join: index-on-index (or index-on-column), shorthand for merge
# concat: stacks DataFrames along axis (no key matching)

# join example (aligns on index)
df_x = pd.DataFrame({'A': [1,2,3]}, index=['a','b','c'])
df_y = pd.DataFrame({'B': [4,5,6]}, index=['a','b','d'])  # 'd' not in df_x

print('join (left by default):')
print(df_x.join(df_y, how='outer'))

# concat
print('\nconcat axis=0 (stack rows):')
print(pd.concat([df_x, df_x], axis=0))

print('\nconcat axis=1 (stack cols):')
print(pd.concat([df_x, df_y], axis=1))

join (left by default):
     A    B
a  1.0  4.0
b  2.0  5.0
c  3.0  NaN
d  NaN  6.0

concat axis=0 (stack rows):
   A
a  1
b  2
c  3
a  1
b  2
c  3

concat axis=1 (stack cols):
     A    B
a  1.0  4.0
b  2.0  5.0
c  3.0  NaN
d  NaN  6.0


> **INTERVIEW QUESTION:** What is the difference between `merge` and `join`? When would you use `concat`?
>
> **Answer:**
> - `merge`: full SQL-style join on columns or index. Explicitly specify left/right keys. Most flexible.
> - `join`: convenience method that joins on the index by default (or on a column vs index). Equivalent to `merge(..., left_index=True, right_index=True)` by default.
> - `concat`: stacks DataFrames along an axis without key-matching logic. Use for appending rows (`axis=0`) or combining same-indexed DataFrames side-by-side (`axis=1`).
>
> **PANDAS GOTCHA:** `validate='m:1'` (or `'1:1'`, `'1:m'`) is a production safety net — it raises `MergeError` if the join key has unexpected duplicates. Always use it in ETL pipelines.

---
## 7. Window Functions — rolling, expanding, ewm

In [44]:
# Realistic time series: daily stock prices
np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=60, freq='B')  # business days
price = 100 + np.cumsum(np.random.randn(60) * 2)
volume = np.random.randint(1_000_000, 5_000_000, 60)

df_stock = pd.DataFrame({'price': price, 'volume': volume}, index=dates)
print(df_stock.head(8).round(2))

             price   volume
2023-01-02  100.99  4521441
2023-01-03  100.72  3935840
2023-01-04  102.01  1351279
2023-01-05  105.06  4011062
2023-01-06  104.59  2870230
2023-01-09  104.12  4747389
2023-01-10  107.28  4793700
2023-01-11  108.82  1489570


In [45]:
# --- rolling: fixed window ---
df_stock['MA_5']  = df_stock['price'].rolling(window=5).mean()   # 5-day MA
df_stock['MA_20'] = df_stock['price'].rolling(window=20).mean()  # 20-day MA
df_stock['vol_20'] = df_stock['price'].rolling(window=20).std()  # 20-day volatility

print('Rolling stats (first 25 rows):')
print(df_stock[['price','MA_5','MA_20','vol_20']].head(25).round(2))
print(f'\nNaN count in MA_20: {df_stock["MA_20"].isna().sum()} (first 19 rows have insufficient data)')

Rolling stats (first 25 rows):
             price    MA_5   MA_20  vol_20
2023-01-02  100.99     NaN     NaN     NaN
2023-01-03  100.72     NaN     NaN     NaN
2023-01-04  102.01     NaN     NaN     NaN
2023-01-05  105.06     NaN     NaN     NaN
2023-01-06  104.59  102.67     NaN     NaN
2023-01-09  104.12  103.30     NaN     NaN
2023-01-10  107.28  104.61     NaN     NaN
2023-01-11  108.82  105.97     NaN     NaN
2023-01-12  107.88  106.54     NaN     NaN
2023-01-13  108.96  107.41     NaN     NaN
2023-01-16  108.03  108.19     NaN     NaN
2023-01-17  107.10  108.16     NaN     NaN
2023-01-18  107.59  107.91     NaN     NaN
2023-01-19  103.76  107.09     NaN     NaN
2023-01-20  100.31  105.36     NaN     NaN
2023-01-23   99.19  103.59     NaN     NaN
2023-01-24   97.16  101.60     NaN     NaN
2023-01-25   97.79   99.64     NaN     NaN
2023-01-26   95.97   98.08     NaN     NaN
2023-01-27   93.15   96.65  103.02    4.72
2023-01-30   96.08   96.03  102.78    4.95
2023-01-31   95.63   95

In [46]:
# min_periods — allow partial windows
df_stock['MA_20_partial'] = df_stock['price'].rolling(window=20, min_periods=1).mean()
print('MA_20 with min_periods=1 (no NaN at start):')
print(df_stock[['price','MA_20','MA_20_partial']].head(5).round(2))

MA_20 with min_periods=1 (no NaN at start):
             price  MA_20  MA_20_partial
2023-01-02  100.99    NaN         100.99
2023-01-03  100.72    NaN         100.86
2023-01-04  102.01    NaN         101.24
2023-01-05  105.06    NaN         102.20
2023-01-06  104.59    NaN         102.67


In [47]:
# --- expanding: growing window (all data up to current point) ---
df_stock['cum_max']  = df_stock['price'].expanding().max()   # all-time high
df_stock['cum_mean'] = df_stock['price'].expanding().mean()  # cumulative mean

print('Expanding window (cumulative):')
print(df_stock[['price','cum_max','cum_mean']].head(10).round(2))

Expanding window (cumulative):
             price  cum_max  cum_mean
2023-01-02  100.99   100.99    100.99
2023-01-03  100.72   100.99    100.86
2023-01-04  102.01   102.01    101.24
2023-01-05  105.06   105.06    102.20
2023-01-06  104.59   105.06    102.67
2023-01-09  104.12   105.06    102.92
2023-01-10  107.28   107.28    103.54
2023-01-11  108.82   108.82    104.20
2023-01-12  107.88   108.82    104.61
2023-01-13  108.96   108.96    105.04


In [48]:
# --- ewm: exponentially weighted moving average ---
# More weight on recent observations; no hard cutoff like rolling
df_stock['EWM_12'] = df_stock['price'].ewm(span=12, adjust=False).mean()  # fast EMA
df_stock['EWM_26'] = df_stock['price'].ewm(span=26, adjust=False).mean()  # slow EMA
df_stock['MACD']   = df_stock['EWM_12'] - df_stock['EWM_26']              # MACD signal

print('EWM and MACD (last 10 rows):')
print(df_stock[['price','EWM_12','EWM_26','MACD']].tail(10).round(3))

EWM and MACD (last 10 rows):
             price  EWM_12  EWM_26   MACD
2023-03-13  78.101  81.132  85.107 -3.975
2023-03-14  77.331  80.547  84.531 -3.984
2023-03-15  75.977  79.844  83.897 -4.053
2023-03-16  77.200  79.437  83.401 -3.964
2023-03-17  79.262  79.410  83.095 -3.684
2023-03-20  81.125  79.674  82.949 -3.275
2023-03-21  79.446  79.639  82.689 -3.050
2023-03-22  78.828  79.514  82.403 -2.889
2023-03-23  79.490  79.511  82.187 -2.677
2023-03-24  81.441  79.808  82.132 -2.325


In [49]:
# --- rolling with custom functions ---
# Bollinger Bands: MA ± 2*std
df_stock['BB_upper'] = df_stock['MA_20'] + 2 * df_stock['vol_20']
df_stock['BB_lower'] = df_stock['MA_20'] - 2 * df_stock['vol_20']

df_stock['in_band'] = (
    (df_stock['price'] >= df_stock['BB_lower']) &
    (df_stock['price'] <= df_stock['BB_upper'])
)

print(f'Price within Bollinger Bands: {df_stock["in_band"].sum()} / {df_stock["in_band"].count()} days')

Price within Bollinger Bands: 38 / 60 days


In [50]:
# --- Window functions on grouped data ---
df_multi = pd.DataFrame({
    'ticker': ['AAPL']*5 + ['GOOG']*5,
    'price':  [150,152,151,155,158, 100,102,101,103,106]
})

# Per-ticker 3-day MA
df_multi['MA_3'] = df_multi.groupby('ticker')['price'].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)
print('Per-group rolling window:')
print(df_multi)

Per-group rolling window:
  ticker  price        MA_3
0   AAPL    150  150.000000
1   AAPL    152  151.000000
2   AAPL    151  151.000000
3   AAPL    155  152.666667
4   AAPL    158  154.666667
5   GOOG    100  100.000000
6   GOOG    102  101.000000
7   GOOG    101  101.000000
8   GOOG    103  102.000000
9   GOOG    106  103.333333


> **INTERVIEW QUESTION:** What is the difference between `rolling`, `expanding`, and `ewm`?
>
> **Answer:**
> - `rolling(n)`: fixed-size window of last N observations. First N-1 values are `NaN` by default.
> - `expanding()`: window grows from the start — equivalent to `rolling(min_periods=1)` that expands. Computes cumulative statistics.
> - `ewm(span=n)`: exponentially weighted — recent observations get higher weight, older ones decay exponentially. No hard window cutoff, so uses all historical data.
>
> **PANDAS GOTCHA:** `rolling` on a grouped DataFrame resets the window at each group boundary only if you use `groupby().transform(lambda x: x.rolling(...))`. If you apply rolling to the whole DataFrame first, it crosses group boundaries.

---
## 8. Memory Optimization

In [51]:
# Create a DataFrame that wastes memory
np.random.seed(0)
N = 100_000

df_mem = pd.DataFrame({
    'user_id':   np.random.randint(1, 10_000, N),          # int64
    'age':       np.random.randint(18, 80, N),              # int64 (only needs int8)
    'score':     np.random.uniform(0, 100, N),              # float64
    'category':  np.random.choice(['A','B','C','D'], N),    # object (should be category)
    'region':    np.random.choice(['North','South','East','West'], N),  # object
    'flag':      np.random.choice([True, False], N),        # bool
    'small_int': np.random.randint(0, 5, N)                 # int64 (only needs int8)
})

def mem_usage_mb(df):
    return df.memory_usage(deep=True).sum() / 1024**2

print(f'Original memory: {mem_usage_mb(df_mem):.2f} MB')
print()
print(df_mem.dtypes)

Original memory: 13.02 MB

user_id        int64
age            int64
score        float64
category         str
region           str
flag            bool
small_int      int64
dtype: object


In [52]:
# --- Downcast integers ---
df_opt = df_mem.copy()

# int64 → smallest fitting int type
for col in ['user_id', 'age', 'small_int']:
    col_min, col_max = df_opt[col].min(), df_opt[col].max()
    print(f'{col}: [{col_min}, {col_max}]')
    df_opt[col] = pd.to_numeric(df_opt[col], downcast='integer')
    print(f'  → {df_opt[col].dtype}')

user_id: [1, 9999]
  → int16
age: [18, 79]
  → int8
small_int: [0, 4]
  → int8


In [53]:
# --- Downcast float ---
df_opt['score'] = pd.to_numeric(df_opt['score'], downcast='float')
print(f'score dtype after downcast: {df_opt["score"].dtype}')  # float32

score dtype after downcast: float32


In [54]:
# --- Convert low-cardinality strings to category ---
print('\nUnique values:')
print(f'category: {df_opt["category"].nunique()} unique / {len(df_opt)} rows')
print(f'region:   {df_opt["region"].nunique()} unique / {len(df_opt)} rows')

df_opt['category'] = df_opt['category'].astype('category')
df_opt['region']   = df_opt['region'].astype('category')

print(f'\nAfter category conversion:')
print(df_opt[['category','region']].dtypes)


Unique values:


category: 4 unique / 100000 rows
region:   4 unique / 100000 rows



After category conversion:
category    category
region      category
dtype: object


In [55]:
# --- memory_usage(deep=True) ---
print('Memory per column (before vs after):')
before = df_mem.memory_usage(deep=True)
after  = df_opt.memory_usage(deep=True)
comp = pd.DataFrame({'before_KB': (before/1024).round(1),
                     'after_KB':  (after/1024).round(1)})
comp['savings_%'] = ((1 - comp['after_KB']/comp['before_KB'])*100).round(1)
print(comp)
print(f'\nTotal: {mem_usage_mb(df_mem):.2f} MB → {mem_usage_mb(df_opt):.2f} MB')
print(f'Reduction: {(1 - mem_usage_mb(df_opt)/mem_usage_mb(df_mem))*100:.1f}%')

Memory per column (before vs after):


           before_KB  after_KB  savings_%
Index            0.1       0.1        0.0
user_id        781.2     195.3       75.0
age            781.2      97.7       87.5
score          781.2     390.6       50.0
category      4882.8      97.9       98.0
region        5224.8      97.9       98.1
flag            97.7      97.7        0.0
small_int      781.2      97.7       87.5



Total: 13.02 MB → 1.05 MB
Reduction: 91.9%


In [56]:
# --- category dtype: bonus benefits ---
# 1. Faster groupby
import time
t1 = time.perf_counter()
for _ in range(100): df_mem.groupby('region')['score'].mean()
t2 = time.perf_counter()
for _ in range(100): df_opt.groupby('region')['score'].mean()
t3 = time.perf_counter()
print(f'groupby on object: {(t2-t1)*10:.1f}ms  vs  category: {(t3-t2)*10:.1f}ms')

# 2. Categorical has ordered option — useful for ordinal data
size_cat = pd.Categorical(['S','M','L','XL','M','S'], 
                          categories=['S','M','L','XL'], 
                          ordered=True)
print('\nOrdered category comparisons:')
print(pd.Series(size_cat) > 'M')  # vectorized comparison using order

groupby on object: 11.7ms  vs  category: 2.5ms

Ordered category comparisons:
0    False
1    False
2     True
3     True
4    False
5    False
dtype: bool


> **INTERVIEW QUESTION:** How would you reduce the memory footprint of a large Pandas DataFrame?
>
> **Answer:**
> 1. **Downcast numerics**: `pd.to_numeric(col, downcast='integer'/'float')` — e.g., int64→int8 saves 87.5%
> 2. **category dtype**: for columns with low cardinality (< ~50% unique values) — stores codes (integers) instead of repeated strings
> 3. **float32 vs float64**: halves float memory with minor precision loss
> 4. **Sparse arrays**: `pd.arrays.SparseArray` for columns with many repeated values (e.g., mostly zeros)
> 5. **Read in chunks**: `pd.read_csv(..., chunksize=N)` for files too large for RAM
>
> **PANDAS GOTCHA:** `memory_usage()` without `deep=True` underreports object columns — it shows pointer size only, not actual string memory. Always use `deep=True` for accurate measurements.

---
## 9. pd.cut vs pd.qcut

In [57]:
np.random.seed(42)
ages = np.random.exponential(scale=20, size=1000).clip(0, 90).astype(int)
s_ages = pd.Series(ages, name='age')

print('Age distribution summary:')
print(s_ages.describe())

Age distribution summary:
count    1000.00000
mean       18.81000
std        18.78672
min         0.00000
25%         5.00000
50%        13.00000
75%        27.00000
max        90.00000
Name: age, dtype: float64


In [58]:
# --- pd.cut: EQUAL WIDTH bins ---
# You define the bin edges (or number of bins → equal width)
cut_result = pd.cut(s_ages, bins=[0, 18, 35, 50, 65, 90],
                    labels=['0-18','19-35','36-50','51-65','66-90'],
                    right=True)   # (left, right] intervals

print('pd.cut (equal width — fixed bin edges):')
print(cut_result.value_counts().sort_index())
print('\nNote: bins are unequally populated (skewed data)')

pd.cut (equal width — fixed bin edges):
age
0-18     572
19-35    210
36-50     88
51-65     43
66-90     33
Name: count, dtype: int64

Note: bins are unequally populated (skewed data)


In [59]:
# --- pd.qcut: EQUAL FREQUENCY bins ---
# Each bin contains approximately the same number of observations
qcut_result = pd.qcut(s_ages, q=5,
                       labels=['Q1','Q2','Q3','Q4','Q5'],
                       duplicates='drop')  # handle duplicate bin edges

print('pd.qcut (equal frequency — quantile-based):')
print(qcut_result.value_counts().sort_index())
print('\nNote: each bin has ~200 observations')

# Inspect actual bin edges
qcut_with_edges = pd.qcut(s_ages, q=5, duplicates='drop')
print('\nBin edges:', qcut_with_edges.cat.categories.tolist())

pd.qcut (equal frequency — quantile-based):
age
Q1    205
Q2    209
Q3    190
Q4    204
Q5    192
Name: count, dtype: int64

Note: each bin has ~200 observations

Bin edges: [Interval(-0.001, 3.0, closed='right'), Interval(3.0, 9.0, closed='right'), Interval(9.0, 17.0, closed='right'), Interval(17.0, 32.0, closed='right'), Interval(32.0, 90.0, closed='right')]


In [60]:
# --- Practical example: credit risk scoring ---
credit_scores = pd.Series(
    np.random.normal(650, 100, 500).clip(300, 850).astype(int),
    name='credit_score'
)

# Equal width: industry standard ranges
rating = pd.cut(credit_scores,
                bins=[299, 579, 669, 739, 799, 851],
                labels=['Poor','Fair','Good','Very Good','Exceptional'])

print('Credit ratings (fixed industry bins):')
print(rating.value_counts().sort_index())

# Equal frequency: internal quartile-based segmentation
segment = pd.qcut(credit_scores, q=4, labels=['Bottom25','Lower50','Upper75','Top'])
print('\nInternal quartile segments:')
print(segment.value_counts().sort_index())

Credit ratings (fixed industry bins):
credit_score
Poor            95
Fair           172
Good           126
Very Good       62
Exceptional     45
Name: count, dtype: int64

Internal quartile segments:
credit_score
Bottom25    129
Lower50     122
Upper75     124
Top         125
Name: count, dtype: int64


In [61]:
# --- retbins parameter: get the bin edges back ---
result, bins = pd.cut(s_ages, bins=4, retbins=True)
print('Automatically computed equal-width bin edges:')
print(bins.round(1))
print(result.value_counts().sort_index())

Automatically computed equal-width bin edges:
[-0.1 22.5 45.  67.5 90. ]
age
(-0.09, 22.5]    697
(22.5, 45.0]     203
(45.0, 67.5]      71
(67.5, 90.0]      29
Name: count, dtype: int64


> **INTERVIEW QUESTION:** What is the difference between `pd.cut` and `pd.qcut`, and when would you use each?
>
> **Answer:**
> - `pd.cut`: **equal-width** bins — you specify bin edges or a count of equal-width intervals. Use when domain knowledge defines meaningful ranges (e.g., age groups 0-18, 19-35; credit score tiers).
> - `pd.qcut`: **equal-frequency** (quantile-based) bins — each bin has approximately the same number of observations. Use when you want balanced classes for ML, or to avoid class imbalance in analysis.
>
> **PANDAS GOTCHA:** `pd.qcut` raises `ValueError` if there are duplicate bin edges (common with heavily skewed/discrete data). Fix with `duplicates='drop'`.

---
## 10. pipe() Method — Method Chaining and Pipelines

In [62]:
# Without pipe: deeply nested or intermediate variables
raw = pd.DataFrame({
    'name':    ['Alice', 'Bob', None, 'Dave', 'Eve'],
    'age':     [25, None, 35, 45, 28],
    'salary':  [70000, 80000, 90000, None, 65000],
    'dept':    ['eng', 'ENG', 'Hr', 'HR', 'eng']
})

# Ugly nested approach:
# result = drop_dupes(fill_missing(normalize_dept(raw)))
# Intermediate variable approach pollutes namespace:
step1 = raw.dropna()
step2 = step1.copy(); step2['dept'] = step2['dept'].str.upper()
step3 = step2[step2['salary'] > 70000]
print('Intermediate variable approach:')
print(step3)

Intermediate variable approach:
Empty DataFrame
Columns: [name, age, salary, dept]
Index: []


In [63]:
# With pipe: readable left-to-right pipeline
def fill_age(df):
    """Fill missing age with median"""
    df = df.copy()
    df['age'] = df['age'].fillna(df['age'].median())
    return df

def normalize_dept(df):
    """Uppercase department names"""
    df = df.copy()
    df['dept'] = df['dept'].str.upper()
    return df

def add_seniority(df, threshold=75000):
    """Add seniority flag based on salary threshold"""
    df = df.copy()
    df['senior'] = df['salary'] > threshold
    return df

def filter_complete(df):
    """Drop rows with any NaN"""
    return df.dropna()

# Clean, readable pipeline
result = (
    raw
    .pipe(fill_age)
    .pipe(normalize_dept)
    .pipe(filter_complete)       # drops rows with NaN salary (name=None already dropped)
    .pipe(add_seniority, threshold=75000)   # pass extra arg
    .reset_index(drop=True)
)

print('pipe() pipeline result:')
print(result)

pipe() pipeline result:
    name   age   salary dept  senior
0  Alice  25.0  70000.0  ENG   False
1    Bob  31.5  80000.0  ENG    True
2    Eve  28.0  65000.0  ENG   False


In [64]:
# pipe also works with functions that take DataFrame as non-first argument
# pipe(func, *args, **kwargs) where func(df, *args, **kwargs)

def scale_column(df, col, factor):
    df = df.copy()
    df[col] = df[col] * factor
    return df

result2 = (
    raw
    .pipe(fill_age)
    .pipe(filter_complete)
    .pipe(scale_column, col='salary', factor=1.1)  # 10% raise
    .reset_index(drop=True)
)
print('After 10% raise via pipe:')
print(result2[['name', 'salary']])

After 10% raise via pipe:
    name   salary
0  Alice  77000.0
1    Bob  88000.0
2    Eve  71500.0


In [65]:
# pipe with (callable, data_kwarg) for functions where df is NOT first arg
import functools

def merge_lookup(lookup_df, main_df, on):
    """lookup_df is first arg — not the piped DataFrame"""
    return main_df.merge(lookup_df, on=on, how='left')

lookup = pd.DataFrame({'dept': ['ENG','HR'], 'budget': [1_000_000, 500_000]})

result3 = (
    result
    .pipe(merge_lookup, lookup, on='dept')
)
print('After merging budget lookup:')
print(result3)

After merging budget lookup:
  dept   budget   name   age   salary senior
0  ENG  1000000  Alice  25.0  70000.0  False
1  ENG  1000000    Bob  31.5  80000.0   True
2  ENG  1000000    Eve  28.0  65000.0  False
3   HR   500000    NaN   NaN      NaN    NaN


> **INTERVIEW QUESTION:** What is the benefit of using `pipe()` over assigning intermediate variables?
>
> **Answer:** `pipe()` enables **method chaining** — a left-to-right readable flow that mirrors how you think about the transformation sequence. Benefits:
> 1. No intermediate variable pollution
> 2. Each step is a named, testable function
> 3. Easy to insert/remove/reorder steps
> 4. Works naturally with Pandas' fluent API
> 5. Pipelines can be composed and reused
>
> It mirrors sklearn's `Pipeline` philosophy applied to data cleaning/feature engineering.

---
## 11. eval() and query()

In [66]:
np.random.seed(0)
N = 500_000
df_eval = pd.DataFrame({
    'A': np.random.randn(N),
    'B': np.random.randn(N),
    'C': np.random.randn(N),
    'D': np.random.randint(0, 4, N),
    'label': np.random.choice(['alpha', 'beta', 'gamma'], N)
})

In [67]:
import timeit

# --- eval() for column arithmetic expressions ---
# Standard approach: creates 3 large intermediate arrays
t_std = timeit.timeit(
    lambda: df_eval['A'] + df_eval['B'] + df_eval['C'] * df_eval['D'],
    number=20
)

# eval: expression parsed and evaluated in a single pass (uses numexpr under the hood)
t_eval = timeit.timeit(
    lambda: df_eval.eval('A + B + C * D'),
    number=20
)

print(f'Standard:   {t_std*1000/20:.1f}ms')
print(f'eval():     {t_eval*1000/20:.1f}ms')
print('(eval is faster on large DataFrames due to reduced memory allocation)')

Standard:   2.0ms
eval():     5.7ms
(eval is faster on large DataFrames due to reduced memory allocation)


In [68]:
# eval can create new columns in-place
df_eval.eval('E = A**2 + B**2', inplace=True)
print('New column E created via eval:')
print(df_eval[['A','B','E']].head(4).round(3))

New column E created via eval:
       A      B      E
0  1.764  1.486  5.321
1  0.400 -0.473  0.384
2  0.979  1.417  2.967
3  2.241  0.885  5.805


In [69]:
# eval with local variable using @
threshold = 1.5
result_eval = df_eval.eval('A > @threshold & B < @threshold')
print(f'Rows where A > {threshold} and B < {threshold}: {result_eval.sum()}')

Rows where A > 1.5 and B < 1.5: 31202


In [70]:
# --- query() for row filtering ---
# Standard
t_std = timeit.timeit(
    lambda: df_eval[(df_eval['A'] > 0) & (df_eval['B'] < 0) & (df_eval['D'] == 2)],
    number=20
)

# query: cleaner syntax, similar performance benefit from numexpr
t_query = timeit.timeit(
    lambda: df_eval.query('A > 0 and B < 0 and D == 2'),
    number=20
)

print(f'Standard filter:  {t_std*1000/20:.1f}ms')
print(f'query():          {t_query*1000/20:.1f}ms')

result_q = df_eval.query('A > 0 and B < 0 and D == 2')
print(f'Matching rows: {len(result_q)}')

Standard filter:  6.6ms
query():          12.7ms
Matching rows: 31335


In [71]:
# query with string columns and @ for local variables
target_label = 'alpha'
result_str = df_eval.query('label == @target_label and A > 0')
print(f'alpha rows with A>0: {len(result_str)}')

# query with list membership (in operator)
result_in = df_eval.query('label in ["alpha", "beta"] and D >= 2')
print(f'alpha/beta rows with D>=2: {len(result_in)}')

alpha rows with A>0: 83493
alpha/beta rows with D>=2: 167119


In [72]:
# Limitations of eval/query
print('eval/query limitations:')
print('1. Cannot call arbitrary Python functions inside expressions')
print('2. Column names with spaces need backtick quoting: `my col`')
print('3. Requires numexpr for maximum speed benefit (auto-detected)')

import numexpr
print(f'\nnumexpr available: True (version {numexpr.__version__})')
print('eval/query will use numexpr for arrays > 10,000 elements')

eval/query limitations:
1. Cannot call arbitrary Python functions inside expressions
2. Column names with spaces need backtick quoting: `my col`
3. Requires numexpr for maximum speed benefit (auto-detected)

numexpr available: True (version 2.14.2)
eval/query will use numexpr for arrays > 10,000 elements


> **INTERVIEW QUESTION:** When should you use `eval()` and `query()` instead of standard Pandas expressions?
>
> **Answer:** For large DataFrames (> ~100k rows), `eval()` and `query()` can be significantly faster because they use `numexpr` to evaluate expressions in a single pass without creating intermediate arrays. They also produce cleaner, more readable code for complex boolean filters. Use them when:
> - Filtering with multiple conditions (query is cleaner than `&`/`|` with parentheses)
> - Computing new columns from expressions involving multiple columns
> - Memory is a concern (no large intermediate arrays)
>
> **PANDAS GOTCHA:** Columns with spaces in their names must be wrapped in backticks: `df.query('\`my col\` > 5')`. Also, you cannot call arbitrary Python functions inside eval/query expressions.

---
## 12. Handling Datetime

In [73]:
# --- pd.to_datetime: flexible parsing ---
date_strings = ['2023-01-15', '01/15/2023', 'Jan 15, 2023', '15-Jan-23', '20230115']
dates = pd.to_datetime(date_strings, format='mixed')
print('Parsed dates:')
for orig, parsed in zip(date_strings, dates):
    print(f'  {orig:20s} → {parsed}')

Parsed dates:
  2023-01-15           → 2023-01-15 00:00:00
  01/15/2023           → 2023-01-15 00:00:00
  Jan 15, 2023         → 2023-01-15 00:00:00
  15-Jan-23            → 2023-01-15 00:00:00
  20230115             → 2023-01-15 00:00:00


In [74]:
# format= parameter for speed on large datasets
import time

N = 100_000
date_list = ['2023-01-15 14:30:00'] * N

t1 = time.perf_counter()
pd.to_datetime(date_list)                            # infer format
t2 = time.perf_counter()
pd.to_datetime(date_list, format='%Y-%m-%d %H:%M:%S')  # explicit format
t3 = time.perf_counter()

print(f'Without format: {(t2-t1)*1000:.1f}ms')
print(f'With format:    {(t3-t2)*1000:.1f}ms')
print('Explicit format is faster on large data (no format inference)')

Without format: 44.9ms
With format:    40.6ms
Explicit format is faster on large data (no format inference)


In [75]:
# --- .dt accessor ---
np.random.seed(0)
dates = pd.date_range('2022-01-01', '2023-12-31', freq='D')
df_dt = pd.DataFrame({
    'date':    dates,
    'revenue': np.random.randint(1000, 10000, len(dates))
})

# Extract components
df_dt['year']    = df_dt['date'].dt.year
df_dt['month']   = df_dt['date'].dt.month
df_dt['day']     = df_dt['date'].dt.day
df_dt['weekday'] = df_dt['date'].dt.day_name()    # 'Monday', 'Tuesday', ...
df_dt['quarter'] = df_dt['date'].dt.quarter
df_dt['week']    = df_dt['date'].dt.isocalendar().week
df_dt['is_wknd'] = df_dt['date'].dt.dayofweek >= 5  # 5=Sat, 6=Sun

print(df_dt.head(5))

        date  revenue  year  month  day    weekday  quarter  week  is_wknd
0 2022-01-01     3732  2022      1    1   Saturday        1    52     True
1 2022-01-02     4264  2022      1    2     Sunday        1    52     True
2 2022-01-03     5859  2022      1    3     Monday        1     1    False
3 2022-01-04     8891  2022      1    4    Tuesday        1     1    False
4 2022-01-05     5373  2022      1    5  Wednesday        1     1    False


In [76]:
# --- resample: time-based groupby ---
df_dt_indexed = df_dt.set_index('date')

print('Weekly total revenue (W = week ending Sunday):')
print(df_dt_indexed['revenue'].resample('W').sum().head(5))

print('\nMonthly stats:')
monthly = df_dt_indexed['revenue'].resample('ME').agg(['sum','mean','max'])
print(monthly.head(6).round(1))

print('\nQuarterly average:')
print(df_dt_indexed['revenue'].resample('QE').mean().round(1))

Weekly total revenue (W = week ending Sunday):
date
2022-01-02     7996
2022-01-09    40914
2022-01-16    36160
2022-01-23    36916
2022-01-30    30816
Freq: W-SUN, Name: revenue, dtype: int64

Monthly stats:
               sum    mean   max
date                            
2022-01-31  162417  5239.3  9615
2022-02-28  169221  6043.6  9994
2022-03-31  186422  6013.6  9752
2022-04-30  180878  6029.3  9829
2022-05-31  145381  4689.7  9717
2022-06-30  153574  5119.1  9962

Quarterly average:
date
2022-03-31    5756.2
2022-06-30    5272.9
2022-09-30    5609.8
2022-12-31    5534.6
2023-03-31    5874.0
2023-06-30    5880.8
2023-09-30    5584.1
2023-12-31    5937.3
Freq: QE-DEC, Name: revenue, dtype: float64


In [77]:
# --- Timezone handling ---
utc_times = pd.date_range('2023-01-01', periods=5, freq='h', tz='UTC')
print('UTC times:')
print(utc_times)

# Convert to different timezone
us_east = utc_times.tz_convert('US/Eastern')
print('\nUS Eastern:')
print(us_east)

# Localize naive datetime
naive = pd.Timestamp('2023-06-01 12:00:00')
localized = naive.tz_localize('US/Pacific')
print(f'\nNaive: {naive}  →  Localized: {localized}')
print(f'UTC:   {localized.tz_convert("UTC")}')

UTC times:
DatetimeIndex(['2023-01-01 00:00:00+00:00', '2023-01-01 01:00:00+00:00',
               '2023-01-01 02:00:00+00:00', '2023-01-01 03:00:00+00:00',
               '2023-01-01 04:00:00+00:00'],
              dtype='datetime64[us, UTC]', freq='h')

US Eastern:
DatetimeIndex(['2022-12-31 19:00:00-05:00', '2022-12-31 20:00:00-05:00',
               '2022-12-31 21:00:00-05:00', '2022-12-31 22:00:00-05:00',
               '2022-12-31 23:00:00-05:00'],
              dtype='datetime64[us, US/Eastern]', freq='h')

Naive: 2023-06-01 12:00:00  →  Localized: 2023-06-01 12:00:00-07:00
UTC:   2023-06-01 19:00:00+00:00


In [78]:
# --- Period vs Timestamp ---
# Timestamp: point in time
# Period: duration / interval

p = pd.Period('2023-Q2')
print(f'Period: {p}')
print(f'Start:  {p.start_time}')
print(f'End:    {p.end_time}')

# Converting between Period and Timestamp
ts = pd.Timestamp('2023-05-15')
print(f'\nTimestamp to monthly Period: {ts.to_period("M")}')
print(f'Period to Timestamp: {pd.Period("2023-05", freq="M").to_timestamp()}')

Period: 2023Q2
Start:  2023-04-01 00:00:00
End:    2023-06-30 23:59:59.999999

Timestamp to monthly Period: 2023-05
Period to Timestamp: 2023-05-01 00:00:00


In [79]:
# --- Time deltas and date arithmetic ---
df_events = pd.DataFrame({
    'start': pd.to_datetime(['2023-01-10', '2023-02-15', '2023-03-01']),
    'end':   pd.to_datetime(['2023-01-25', '2023-03-10', '2023-03-15'])
})

df_events['duration_days'] = (df_events['end'] - df_events['start']).dt.days
df_events['end_month']     = df_events['end'].dt.to_period('M')
df_events['days_from_now'] = (pd.Timestamp.today() - df_events['start']).dt.days

print(df_events)

       start        end  duration_days end_month  days_from_now
0 2023-01-10 2023-01-25             15   2023-01           1291
1 2023-02-15 2023-03-10             23   2023-03           1255
2 2023-03-01 2023-03-15             14   2023-03           1241


> **INTERVIEW QUESTION:** How does `resample()` differ from `groupby()` for time series data?
>
> **Answer:** Both aggregate data into groups, but:
> - `resample()` is specifically for **time-based resampling** — the index must be a `DatetimeIndex`. It understands calendar semantics (weeks end on Sunday, months have variable lengths, etc.).
> - `groupby()` uses explicit column values and has no time awareness.
>
> Use `resample()` when working with time series (daily → monthly → quarterly aggregations). Common aliases: `'D'` (day), `'W'` (week), `'ME'` (month end), `'QE'` (quarter end), `'YE'` (year end).
>
> **PANDAS GOTCHA:** `tz_localize` and `tz_convert` are different: `localize` adds timezone info to a naive datetime (use once), `convert` changes from one timezone to another (requires existing timezone info).

---
## 13. explode(), melt(), wide_to_long()

In [80]:
# --- explode(): one row per element in list-like cells ---
df_tags = pd.DataFrame({
    'user_id': [1, 2, 3],
    'name':    ['Alice', 'Bob', 'Carol'],
    'tags':    [['python', 'pandas'], ['sql', 'python', 'spark'], ['r', 'ggplot']]
})

print('Before explode:')
print(df_tags)

df_exploded = df_tags.explode('tags').reset_index(drop=True)
print('\nAfter explode:')
print(df_exploded)

Before explode:
   user_id   name                  tags
0        1  Alice      [python, pandas]
1        2    Bob  [sql, python, spark]
2        3  Carol           [r, ggplot]

After explode:
   user_id   name    tags
0        1  Alice  python
1        1  Alice  pandas
2        2    Bob     sql
3        2    Bob  python
4        2    Bob   spark
5        3  Carol       r
6        3  Carol  ggplot


In [81]:
# explode + value_counts: find most popular skills
print('Most popular skills:')
print(df_exploded['tags'].value_counts())

# Real use case: explode comma-separated string
df_csv = pd.DataFrame({
    'id':   [1, 2],
    'cats': ['A,B,C', 'B,D']
})
df_csv['cats'] = df_csv['cats'].str.split(',')
print('\nExplode from comma-separated:')
print(df_csv.explode('cats'))

Most popular skills:
tags
python    2
pandas    1
sql       1
spark     1
r         1
ggplot    1
Name: count, dtype: int64

Explode from comma-separated:
   id cats
0   1    A
0   1    B
0   1    C
1   2    B
1   2    D


In [82]:
# --- melt(): wide to long format ---
df_wide = pd.DataFrame({
    'student': ['Alice', 'Bob', 'Carol'],
    'math':    [90, 85, 92],
    'science': [88, 79, 95],
    'english': [75, 88, 83]
})

print('Wide format:')
print(df_wide)

df_long = df_wide.melt(
    id_vars='student',           # columns to keep as identifiers
    value_vars=['math','science','english'],  # columns to unpivot
    var_name='subject',          # name for the variable column
    value_name='score'           # name for the value column
)

print('\nLong format (after melt):')
print(df_long.sort_values(['student','subject']))

Wide format:
  student  math  science  english
0   Alice    90       88       75
1     Bob    85       79       88
2   Carol    92       95       83

Long format (after melt):
  student  subject  score
6   Alice  english     75
0   Alice     math     90
3   Alice  science     88
7     Bob  english     88
1     Bob     math     85
4     Bob  science     79
8   Carol  english     83
2   Carol     math     92
5   Carol  science     95


In [83]:
# melt is the inverse of pivot_table
df_pivot = df_long.pivot_table(index='student', columns='subject', values='score')
df_pivot.columns.name = None
df_pivot = df_pivot.reset_index()
print('Pivot back to wide:')
print(df_pivot)

Pivot back to wide:
  student  english  math  science
0   Alice     75.0  90.0     88.0
1     Bob     88.0  85.0     79.0
2   Carol     83.0  92.0     95.0


In [84]:
# --- wide_to_long(): for stubbed column names like score_2021, score_2022 ---
df_stub = pd.DataFrame({
    'id':         [1, 2, 3],
    'name':       ['A', 'B', 'C'],
    'revenue_2021': [100, 200, 300],
    'revenue_2022': [110, 210, 310],
    'cost_2021':    [50, 80, 120],
    'cost_2022':    [55, 85, 125]
})

print('Wide with year-suffixed columns:')
print(df_stub)

df_wtl = pd.wide_to_long(
    df_stub,
    stubnames=['revenue', 'cost'],  # column prefixes
    i=['id', 'name'],               # id columns
    j='year',                       # new variable column
    sep='_'                         # separator between stub and suffix
).reset_index()

print('\nAfter wide_to_long:')
print(df_wtl.sort_values(['id','year']))

Wide with year-suffixed columns:
   id name  revenue_2021  revenue_2022  cost_2021  cost_2022
0   1    A           100           110         50         55
1   2    B           200           210         80         85
2   3    C           300           310        120        125

After wide_to_long:
   id name  year  revenue  cost
0   1    A  2021      100    50
1   1    A  2022      110    55
2   2    B  2021      200    80
3   2    B  2022      210    85
4   3    C  2021      300   120
5   3    C  2022      310   125


> **INTERVIEW QUESTION:** When would you use `melt()` vs `wide_to_long()`?
>
> **Answer:**
> - `melt()`: general-purpose wide-to-long. Use when column names don't follow a stub pattern, or when you only have one group of value columns.
> - `wide_to_long()`: for columns with a common prefix + suffix pattern (e.g., `revenue_2021`, `revenue_2022`, `cost_2021`). It handles multiple variable groups simultaneously, which would require multiple `melt` calls.
>
> **PANDAS GOTCHA:** `explode()` preserves the original index by default — you'll see duplicate index values. Always follow with `.reset_index(drop=True)` unless you specifically need the original index for alignment.

---
## 14. Duplicates Deep Dive

In [85]:
df_dup = pd.DataFrame({
    'id':      [1, 2, 2, 3, 3, 3, 4],
    'name':    ['Alice', 'Bob', 'Bob', 'Carol', 'Carol', 'Carol', 'Dave'],
    'email':   ['a@x.com','b@x.com','b@x.com','c@x.com','c@y.com','c@x.com','d@x.com'],
    'score':   [90, 85, 85, 92, 92, 92, 78],
    'ts':      ['2023-01','2023-01','2023-02','2023-01','2023-01','2023-02','2023-01']
})

print(df_dup)

   id   name    email  score       ts
0   1  Alice  a@x.com     90  2023-01
1   2    Bob  b@x.com     85  2023-01
2   2    Bob  b@x.com     85  2023-02
3   3  Carol  c@x.com     92  2023-01
4   3  Carol  c@y.com     92  2023-01
5   3  Carol  c@x.com     92  2023-02
6   4   Dave  d@x.com     78  2023-01


In [86]:
# --- duplicated(): returns boolean mask ---
print('duplicated() — default (all cols, keep=first):')
print(df_dup.duplicated())

print('\nduplication count:', df_dup.duplicated().sum())

duplicated() — default (all cols, keep=first):
0    False
1    False
2    False
3    False
4    False
5    False
6    False
dtype: bool

duplication count: 0


In [87]:
# keep parameter:
# 'first'  — mark all duplicates except first occurrence as True
# 'last'   — mark all except last occurrence as True
# False    — mark ALL occurrences as True

print('keep="first" (default): marks 2nd+ occurrence')
print(df_dup.duplicated(keep='first').tolist())

print('\nkeep="last": marks all but last occurrence')
print(df_dup.duplicated(keep='last').tolist())

print('\nkeep=False: marks all duplicate occurrences')
print(df_dup.duplicated(keep=False).tolist())

keep="first" (default): marks 2nd+ occurrence
[False, False, False, False, False, False, False]

keep="last": marks all but last occurrence
[False, False, False, False, False, False, False]

keep=False: marks all duplicate occurrences
[False, False, False, False, False, False, False]


In [88]:
# subset: check duplicates based on specific columns only
print('Duplicates on [id, name] only:')
print(df_dup.duplicated(subset=['id', 'name'], keep=False))
print()
print('Duplicate rows (id+name match):')
print(df_dup[df_dup.duplicated(subset=['id','name'], keep=False)])

Duplicates on [id, name] only:
0    False
1     True
2     True
3     True
4     True
5     True
6    False
dtype: bool

Duplicate rows (id+name match):
   id   name    email  score       ts
1   2    Bob  b@x.com     85  2023-01
2   2    Bob  b@x.com     85  2023-02
3   3  Carol  c@x.com     92  2023-01
4   3  Carol  c@y.com     92  2023-01
5   3  Carol  c@x.com     92  2023-02


In [89]:
# --- drop_duplicates() ---
print('drop_duplicates() (keep=first, all cols):')
print(df_dup.drop_duplicates())
print()

print('drop_duplicates(subset=["id","name"], keep="last"):')
print(df_dup.drop_duplicates(subset=['id','name'], keep='last'))

drop_duplicates() (keep=first, all cols):


   id   name    email  score       ts
0   1  Alice  a@x.com     90  2023-01
1   2    Bob  b@x.com     85  2023-01
2   2    Bob  b@x.com     85  2023-02
3   3  Carol  c@x.com     92  2023-01
4   3  Carol  c@y.com     92  2023-01
5   3  Carol  c@x.com     92  2023-02
6   4   Dave  d@x.com     78  2023-01

drop_duplicates(subset=["id","name"], keep="last"):
   id   name    email  score       ts
0   1  Alice  a@x.com     90  2023-01
2   2    Bob  b@x.com     85  2023-02
5   3  Carol  c@x.com     92  2023-02
6   4   Dave  d@x.com     78  2023-01


In [90]:
# --- Real-world pattern: find duplicates to INVESTIGATE ---
dupes_all = df_dup[df_dup.duplicated(subset=['id'], keep=False)].sort_values('id')
print('All rows with duplicate id (to investigate conflicts):')
print(dupes_all)

# Keep most recent record per ID
df_deduped = (
    df_dup
    .sort_values('ts', ascending=False)
    .drop_duplicates(subset=['id'], keep='first')
    .sort_values('id')
    .reset_index(drop=True)
)
print('\nDeduplicated (keep latest record):')
print(df_deduped)

All rows with duplicate id (to investigate conflicts):
   id   name    email  score       ts
1   2    Bob  b@x.com     85  2023-01
2   2    Bob  b@x.com     85  2023-02
3   3  Carol  c@x.com     92  2023-01
4   3  Carol  c@y.com     92  2023-01
5   3  Carol  c@x.com     92  2023-02

Deduplicated (keep latest record):
   id   name    email  score       ts
0   1  Alice  a@x.com     90  2023-01
1   2    Bob  b@x.com     85  2023-02
2   3  Carol  c@x.com     92  2023-02
3   4   Dave  d@x.com     78  2023-01


> **INTERVIEW QUESTION:** How do you find all rows involved in duplicates (not just the extra copies)?
>
> **Answer:** Use `duplicated(keep=False)` — this marks **every** occurrence of a duplicate row (not just the 2nd+). This is useful for auditing: `df[df.duplicated(subset=['key'], keep=False)]` returns all rows that have conflicts.
>
> **PANDAS GOTCHA:** `drop_duplicates()` without `subset` compares all columns — a common mistake. If your deduplication key is a specific column (like `user_id`), always specify `subset=['user_id']` to avoid accidentally keeping rows that differ in a timestamp column.

---
## 15. Common Interview Traps

### Trap 1: Never Use iterrows() — Use Vectorized Operations

In [91]:
import time

np.random.seed(42)
N = 10_000
df_trap = pd.DataFrame({
    'a': np.random.randn(N),
    'b': np.random.randn(N),
    'c': np.random.randint(0, 10, N)
})

# BAD: iterrows (Python loop, type coercion, slow)
t1 = time.perf_counter()
result_iter = []
for _, row in df_trap.iterrows():
    result_iter.append(row['a'] * row['b'] + row['c'])
t2 = time.perf_counter()
time_iterrows = t2 - t1

# GOOD: vectorized
t3 = time.perf_counter()
result_vec = df_trap['a'] * df_trap['b'] + df_trap['c']
t4 = time.perf_counter()
time_vec = t4 - t3

print(f'iterrows:   {time_iterrows*1000:.1f}ms')
print(f'vectorized: {time_vec*1000:.2f}ms')
print(f'Speedup:    {time_iterrows/time_vec:.0f}x')
print(f'Results equal: {np.allclose(result_iter, result_vec)}')

iterrows:   278.6ms
vectorized: 0.58ms
Speedup:    482x
Results equal: True


In [92]:
# iterrows also COPIES data and returns Series with object dtype!
df_small = pd.DataFrame({'x': [1, 2, 3], 'y': [4.0, 5.0, 6.0]})
for idx, row in df_small.iterrows():
    print(f'Row {idx} dtypes: x={type(row["x"]).__name__}, y={type(row["y"]).__name__}')
    # x comes back as int64, y as float64 — BUT everything is upcast to object dtype in row!
    print(f'  row.dtype = {row.dtype}')  # object! all values coerced
    break

Row 0 dtypes: x=float64, y=float64
  row.dtype = float64


### Trap 2: Index Alignment in Operations

In [93]:
# Pandas ALIGNS by index before doing arithmetic
s1 = pd.Series([1, 2, 3], index=['a', 'b', 'c'])
s2 = pd.Series([10, 20, 30], index=['b', 'c', 'd'])  # different index!

result = s1 + s2
print('s1 + s2 (index alignment):')
print(result)
print('\na is only in s1 → NaN')
print('d is only in s2 → NaN')
print('b and c are in both → aligned and added')

s1 + s2 (index alignment):
a     NaN
b    12.0
c    23.0
d     NaN
dtype: float64

a is only in s1 → NaN
d is only in s2 → NaN
b and c are in both → aligned and added


In [94]:
# This bites you when you filter a Series and add it back
df_align = pd.DataFrame({'a': [10, 20, 30, 40, 50]})
subset = df_align['a'][df_align['a'] > 15]  # index is [1, 2, 3, 4]
print('Original index: ', df_align.index.tolist())
print('Subset index:   ', subset.index.tolist())

# Safe: use .values to ignore index
print('\nSubset values:', subset.values)

# Dangerous: assigning misaligned Series
df2 = df_align.copy()
df2['scaled'] = pd.Series([100, 200, 300])  # only index 0,1,2 align
print('\nAssigning Series with partial index:')
print(df2)  # rows 3,4 get NaN for 'scaled'

Original index:  [0, 1, 2, 3, 4]
Subset index:    [1, 2, 3, 4]

Subset values: [20 30 40 50]

Assigning Series with partial index:
    a  scaled
0  10   100.0
1  20   200.0
2  30   300.0
3  40     NaN
4  50     NaN


### Trap 3: Modifying a DataFrame During Iteration

In [95]:
# Never modify a DataFrame while iterating over it
df_mod = pd.DataFrame({'val': [1, 2, 3, 4, 5]})

# BAD (and undefined behavior):
# for idx, row in df_mod.iterrows():
#     if row['val'] > 3:
#         df_mod.drop(idx, inplace=True)  # DANGER

# GOOD: filter first, then drop
df_clean = df_mod[df_mod['val'] <= 3].copy()
print('Filtered DataFrame:')
print(df_clean)

Filtered DataFrame:
   val
0    1
1    2
2    3


### Trap 4: Beware of inplace=True

In [96]:
# inplace=True does NOT actually save memory — it creates a copy internally
# and replaces the original. It also returns None, breaking method chaining.

df_ip = pd.DataFrame({'a': [3,1,2], 'b': [6,4,5]})

# This returns None — breaks chaining
result = df_ip.sort_values('a', inplace=True)
print('inplace=True returns:', result)  # None
print('df_ip after sort:')
print(df_ip)

# Method chaining pattern (preferred)
df_ip2 = pd.DataFrame({'a': [3,1,2], 'b': [6,4,5]})
df_ip2 = df_ip2.sort_values('a').reset_index(drop=True)  # reassign
print('\nChained (preferred):')
print(df_ip2)

inplace=True returns: None
df_ip after sort:
   a  b
1  1  4
2  2  5
0  3  6

Chained (preferred):
   a  b
0  1  4
1  2  5
2  3  6


### Trap 5: Chained Comparison Needs Parentheses

In [97]:
s = pd.Series([1, 5, 10, 15, 20])

# WRONG: Python evaluates this as (5 < s) and (s < 15) — wrong 'and'
try:
    # result = 5 < s and s < 15  # Would raise ValueError
    pass
except ValueError as e:
    print(e)

# CORRECT: use & with parentheses
result = (s > 5) & (s < 15)
print('(s > 5) & (s < 15):')
print(result)
print('\nValues:', s[result].tolist())

# Also correct: use .between()
print('\ns.between(5, 15, inclusive="neither"):')
print(s.between(5, 15, inclusive='neither'))

(s > 5) & (s < 15):
0    False
1    False
2     True
3    False
4    False
dtype: bool

Values: [10]

s.between(5, 15, inclusive="neither"):
0    False
1    False
2     True
3    False
4    False
dtype: bool


### Trap 6: Categorical dtype and missing categories

In [98]:
# Categorical groupby includes empty categories!
cat_series = pd.Categorical(['a','b','a'], categories=['a','b','c'])
s_cat = pd.Series(cat_series)

print('value_counts with category (includes "c" with 0):')
print(s_cat.value_counts())

print('\nGroupby on categorical (includes empty group "c"):')
df_cat = pd.DataFrame({'key': s_cat, 'val': [1, 2, 3]})
print(df_cat.groupby('key')['val'].sum())  # 'c' appears with 0

# Fix: use observed=True to skip empty categories
print('\nWith observed=True (skip empty):')
print(df_cat.groupby('key', observed=True)['val'].sum())

value_counts with category (includes "c" with 0):
a    2
b    1
c    0
Name: count, dtype: int64

Groupby on categorical (includes empty group "c"):
key
a    4
b    2
Name: val, dtype: int64

With observed=True (skip empty):
key
a    4
b    2
Name: val, dtype: int64


> **INTERVIEW QUESTION:** Why should you never use `iterrows()` in production Pandas code?
>
> **Answer:** Three reasons:
> 1. **Performance**: `iterrows()` is a Python-level loop — 100-1000x slower than vectorized operations for large DataFrames.
> 2. **Type coercion**: each row is returned as a `pd.Series` with `dtype=object`, which can cause unexpected type conversions.
> 3. **Copy semantics**: modifying `row` inside the loop does NOT modify the original DataFrame.
>
> **Alternatives in order of preference:** vectorized arithmetic > `np.where`/`np.select` > `.str`/`.dt` accessors > `Series.map()` with dict > `DataFrame.apply(axis=1)` > `itertuples()` (faster than iterrows if you must iterate) > `iterrows()` (last resort).
>
> **PANDAS GOTCHA:** When performing arithmetic between two Series/DataFrames with different indexes, Pandas **aligns on index first**. This produces `NaN` where indexes don't match — a silent bug that's hard to trace. Use `.values` or `.to_numpy()` to bypass index alignment when you're sure the data is already aligned.

---
## Quick Reference: Interview Cheat Sheet

### Indexing
| Method | Access type | Scalar speed |
|--------|------------|-------------|
| `[]` | column by label; row by slice | — |
| `.loc[]` | label-based | medium |
| `.iloc[]` | position-based (0-indexed, exclusive right) | medium |
| `.at[]` | label-based scalar | **fastest** |
| `.iat[]` | position-based scalar | **fastest** |

### groupby return shapes
| Method | Output shape | Use for |
|--------|-------------|--------|
| `agg()` | one row per group | summary stats |
| `transform()` | same as input | adding group stats as columns |
| `apply()` | any shape | complex per-group logic |
| `filter()` | subset of input rows | removing groups by predicate |

### Reshaping
| Function | Direction | Notes |
|----------|-----------|-------|
| `melt()` | wide → long | general |
| `pivot_table()` | long → wide | with aggregation |
| `stack()` | columns → row index | MultiIndex result |
| `unstack()` | row index → columns | inverse of stack |
| `explode()` | list cell → multiple rows | |
| `wide_to_long()` | wide → long | stubbed column names |

### Performance tiers (fast → slow)
1. Vectorized NumPy/Pandas ops (C-speed, no Python overhead)
2. `.str` / `.dt` accessors
3. `Series.map(dict)` (hash lookup)
4. `eval()` / `query()` (numexpr, avoids intermediate arrays)
5. `Series.map(function)` / `Series.apply(function)` 
6. `DataFrame.apply(axis=1)` (per-row Python loop)
7. `DataFrame.map()` / `applymap()` (per-cell Python loop)
8. `itertuples()` (named tuple iteration)
9. `iterrows()` (slowest, with type coercion — avoid in production)

In [99]:
print('Notebook complete! All 15 advanced Pandas topics covered.')
print()
print('Key takeaways:')
print('1.  Always use .loc (label) or .iloc (position) — never ambiguous []')
print('2.  Fix SettingWithCopyWarning with .loc or .copy() — never ignore it')
print('3.  Vectorized ops > apply > iterrows — profile before choosing')
print('4.  Use named aggregations in groupby for clean, readable code')
print('5.  MultiIndex: unstack() moves index level to columns')
print('6.  Always use validate= in merge for production ETL')
print('7.  rolling=fixed window, expanding=growing, ewm=exponential decay')
print('8.  category dtype + int downcast = 50-80% memory reduction')
print('9.  cut=fixed bins, qcut=equal-frequency bins')
print('10. pipe() enables readable, composable data pipelines')
print('11. query()/eval() avoid intermediate arrays on large DataFrames')
print('12. tz_localize once, tz_convert to switch; use format= for speed')
print('13. explode() + reset_index(drop=True) for list-column unnesting')
print('14. duplicated(keep=False) to audit ALL duplicate occurrences')
print('15. Index alignment is silent — use .values when indexes may differ')

Notebook complete! All 15 advanced Pandas topics covered.

Key takeaways:
1.  Always use .loc (label) or .iloc (position) — never ambiguous []
2.  Fix SettingWithCopyWarning with .loc or .copy() — never ignore it
3.  Vectorized ops > apply > iterrows — profile before choosing
4.  Use named aggregations in groupby for clean, readable code
5.  MultiIndex: unstack() moves index level to columns
6.  Always use validate= in merge for production ETL
7.  rolling=fixed window, expanding=growing, ewm=exponential decay
8.  category dtype + int downcast = 50-80% memory reduction
9.  cut=fixed bins, qcut=equal-frequency bins
10. pipe() enables readable, composable data pipelines
11. query()/eval() avoid intermediate arrays on large DataFrames
12. tz_localize once, tz_convert to switch; use format= for speed
13. explode() + reset_index(drop=True) for list-column unnesting
14. duplicated(keep=False) to audit ALL duplicate occurrences
15. Index alignment is silent — use .values when indexes may diffe